In [ ]:
# No terminal, uma vez: python -m pip install openpyxl pandas xlrd matplotlib seaborn
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
from pathlib import Path

# Coloque a planilha na mesma pasta do notebook.
path = Path("Tabela_11_Domicilios_sem_presenca_de_conjuge_e_com_filhos.xlsx")
if not path.exists():
    path = Path("/mnt/data/Tabela_11_Domicilios_sem_presenca_de_conjuge_e_com_filhos.xlsx")
if not path.exists():
    raise FileNotFoundError("Planilha não encontrada. Ajuste a variável path.")
print("Planilha:", path)

# Sexo da pessoa responsável pelo domicílio

In [ ]:
# Lê todas as abas da planilha
todas = pd.read_excel(path, sheet_name=None, header=None)
print(list(todas.keys()))

In [ ]:
bruto = pd.read_excel(path, sheet_name="Tabela base do SIDRA 9882", header=None)

In [ ]:
bruto

In [ ]:
# Linhas de dados, excluindo os cabeçalhos e a linha da fonte
indicador_2 = bruto.iloc[5:].copy()
indicador_2 = indicador_2[indicador_2.iloc[:, 0].notna()].copy()
indicador_2 = indicador_2[~indicador_2.iloc[:, 0].astype(str).str.startswith("Fonte:")].copy()

In [ ]:
indicador_2

In [ ]:
# Preenchendo os nomes das colunas
indicador_2.columns = [
    "regiao",
    "total",
    "homens",
    "mulheres",
]


In [ ]:
indicador_2 = indicador_2.reset_index(drop=True)
# Quantidade de domicílios, não horas
for col in indicador_2.columns[1:]:
    indicador_2[col] = pd.to_numeric(indicador_2[col], errors="coerce")
indicador_2

In [ ]:
indicador_2.info()

In [ ]:
indicador_2.describe().round(2)

EXCLUIR AS CIDADES, REPAREM NO PADRÃO, ELAS TEM O () QUE INDICAM O UF

In [ ]:
# Municípios possuem a UF entre parênteses, como "Brasília (DF)"
indicador_2 = indicador_2[~indicador_2["regiao"].str.contains(r"\(", na=False)].copy()
indicador_2

# Trazer a Tabela 1.1.1 (horas de trabalho doméstico) e juntar com indicador_2

In [ ]:
# A Tabela 1.1.1 (horas de trabalho doméstico) NÃO foi anexada.
# Caso a tenha, coloque o arquivo .xls/.xlsx na pasta e informe o nome abaixo.
path_horas = Path("Tabela_1_1_1.xls")
indicador_1 = None
if path_horas.exists():
    bruto_1 = pd.read_excel(path_horas, header=None)
    indicador_1 = bruto_1.iloc[8:41].copy()
    indicador_1.columns = ["uf_regiao", "total", "total_branca", "total_preta_parda",
                           "homem_branca", "homem_preta_parda", "mulher_branca", "mulher_preta_parda"]
    indicador_1 = indicador_1.reset_index(drop=True)
    for col in indicador_1.columns[1:]:
        indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")
    display(indicador_1.head())
else:
    print("Tabela 1.1.1 não anexada: análises de horas serão ignoradas.")

In [ ]:
if indicador_1 is not None:
    indicador1e2 = indicador_2.merge(indicador_1, how="left", left_on="regiao",
                                    right_on="uf_regiao", suffixes=("_domicilios", "_horas"))
    indicador1e2 = indicador1e2.drop(columns=["uf_regiao"])
else:
    indicador1e2 = indicador_2.rename(columns={"total": "total_domicilios"}).copy()

In [ ]:
indicador1e2 = indicador1e2.rename(columns={
    "homens": "resp_domicilio_homens",
    "mulheres": "resp_domicilio_mulheres",
})
indicador1e2

In [ ]:
indicador1e2["pct_resp_mulheres"] = (
    indicador1e2["resp_domicilio_mulheres"] / indicador1e2["total_domicilios"] * 100
)
if indicador_1 is not None:
    indicador1e2["horas_mulher"] = (
        indicador1e2["mulher_branca"] + indicador1e2["mulher_preta_parda"]
    ) / 2
else:
    indicador1e2["horas_mulher"] = float("nan")

indicador1e2[["regiao", "pct_resp_mulheres", "horas_mulher",
             "resp_domicilio_mulheres", "total_domicilios"]].sort_values(
                 "pct_resp_mulheres", ascending=False)

**Interpretação:** `pct_resp_mulheres` representa o percentual de domicílios sem cônjuge e com filhos cujo responsável é mulher. `horas_mulher` só poderá ser calculada com a Tabela 1.1.1 (não anexada); nesse caso, corresponde à média simples entre as horas das mulheres brancas e pretas/pardas. Não é correto interpretar os números como horas sem essa segunda fonte.

In [ ]:
indicador_2["mais_mulheres"] = indicador_2["mulheres"] > indicador_2["homens"]
brasil = indicador_2.loc[indicador_2["regiao"] == "Brasil"].iloc[0]
print("Brasil — domicílios sem cônjuge e com filhos (Censo 2022)")
print(f"  mulheres: {int(brasil.mulheres):,}  ({brasil.mulheres/brasil.total*100:.1f}%)")
print(f"  homens:   {int(brasil.homens):,}  ({brasil.homens/brasil.total*100:.1f}%)")
print(f"  {brasil.mulheres/brasil.homens:.1f} mulheres para cada homem responsável")
print("Em todas as UFs/regiões mulheres > homens?", indicador_2["mais_mulheres"].all())
indicador_2[["regiao", "homens", "mulheres", "mais_mulheres"]]

## Gráficos com matplotlib e seaborn

Visualizamos o recorte **domicílios sem cônjuge e com filhos** e as **horas de afazeres** da Tabela 1.1.1.

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
ufs = indicador1e2[~indicador1e2["regiao"].isin(["Brasil"] + regioes)].copy()
fig, ax = plt.subplots(figsize=(9, 8))
ordem = ufs.sort_values("pct_resp_mulheres", ascending=False)
sns.barplot(data=ordem, y="regiao", x="pct_resp_mulheres", ax=ax, color="teal")
ax.set_title("% de domicílios com responsável mulher (sem cônjuge + filhos)")
ax.set_xlabel("% com responsável mulher")
ax.set_ylabel("UF")
plt.tight_layout()
plt.show()

In [ ]:
reg = indicador_2[indicador_2["regiao"].isin(regioes)].copy()
reg_long = reg.melt(id_vars="regiao", value_vars=["homens", "mulheres"],
                    var_name="sexo", value_name="domicilios")
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=reg_long, x="regiao", y="domicilios", hue="sexo", ax=ax)
ax.set_title("Domicílios sem cônjuge e com filhos, por região e sexo do responsável")
ax.set_xlabel("Região")
ax.set_ylabel("Número de domicílios")
plt.tight_layout()
plt.show()

In [ ]:
if indicador_1 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(ufs["horas_mulher"].dropna(), bins=8, kde=True, ax=axes[0])
    axes[0].set_title("Frequência das horas médias das mulheres (UFs)")
    axes[0].set_xlabel("Horas / semana")
    axes[0].set_ylabel("Número de UFs")
    sns.boxplot(y=ufs["horas_mulher"].dropna(), ax=axes[1])
    axes[1].set_title("Box plot das horas das mulheres por UF")
    axes[1].set_ylabel("Horas / semana")
    plt.tight_layout()
    plt.show()
else:
    print("Gráficos de horas indisponíveis: falta a Tabela 1.1.1.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(y=ufs["pct_resp_mulheres"], ax=ax, color="teal")
ax.set_title("Box plot: % responsável mulher nas UFs")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()

In [ ]:
if indicador_1 is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=ufs, x="pct_resp_mulheres", y="horas_mulher", ax=ax, s=80)
    for _, row in ufs.dropna(subset=["horas_mulher"]).iterrows():
        ax.annotate(row["regiao"], (row["pct_resp_mulheres"], row["horas_mulher"]), fontsize=7)
    ax.set_title("Responsável mulher (%) × horas de afazeres das mulheres")
    ax.set_xlabel("% domicílios com responsável mulher")
    ax.set_ylabel("Horas / semana (mulheres)")
    plt.tight_layout()
    plt.show()
else:
    print("Dispersão indisponível: falta a Tabela 1.1.1.")

In [ ]:
ufs["maioria"] = "Mais mulheres"
ufs.loc[ufs["resp_domicilio_mulheres"] <= ufs["resp_domicilio_homens"], "maioria"] = "Mais homens"
fig, ax = plt.subplots(figsize=(6, 4))
ufs["maioria"].value_counts().plot.bar(ax=ax)
ax.set_title("Frequência de UFs segundo o sexo da maioria dos responsáveis")
ax.set_xlabel("")
ax.set_ylabel("Número de UFs")
plt.tight_layout()
plt.show()